# AutoMod AI — Paint/Wrap Retune v2 (InstructPix2Pix)

**Upload this notebook fresh.** Do not let Gemini rewrite the export cell — the last broken zip was after-only.

| Setting | Value |
|---|---|
| After size | always **512×512** |
| `image_guidance_scale` | **2.0** |
| `guidance_scale` | **8.5** |
| Jobs | 20 paint/wrap only |
| Zip must contain | `NNNN_before.jpg`, `NNNN_after.jpg`, `metadata.json` |

### Steps
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Runtime → Run all
3. Upload **`synthetic-source-pack.zip`** when asked
4. Download **`synthetic-pairs-v2.zip`** — unzip into empty `dataset/synthetic-pairs/`

Settings come from the pack's `settings.json`. Do not type them by hand.

In [ ]:
!pip install -q "diffusers>=0.30.0" transformers accelerate safetensors Pillow numpy

import torch
assert torch.cuda.is_available(), (
    "No GPU. Runtime → Change runtime type → T4 GPU → Save, then Runtime → Run all again."
)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

jobs_existing = [p for p in Path(".").rglob("jobs.json") if "synthetic-source-pack" in str(p)]
if jobs_existing:
    jobs_path = jobs_existing[0]
    print("Using already-unzipped pack at", jobs_path)
else:
    print("Click Choose Files and select synthetic-source-pack.zip")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall(".")
    jobs_path = next(Path(".").rglob("jobs.json"))

PACK_DIR = jobs_path.parent
print("Pack:", PACK_DIR.resolve())

In [ ]:
import json, time
from PIL import Image
from diffusers import StableDiffusionInstructPix2PixPipeline, EulerAncestralDiscreteScheduler

settings_path = PACK_DIR / "settings.json"
settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
print("settings.json:")
print(json.dumps({k: v for k, v in settings.items() if k != "negative_prompt"}, indent=2))

RESOLUTION = int(settings.get("resolution", 512))
STEPS = int(settings.get("num_inference_steps", 20))
IMAGE_GUIDANCE = float(settings.get("image_guidance_scale", 2.0))
GUIDANCE = float(settings.get("guidance_scale", 8.5))
MODEL_ID = "timbrooks/instruct-pix2pix"
NEGATIVE = settings.get(
    "negative_prompt",
    "tinted windows, colored glass, painted windshield, red glass, blue glass, green glass, "
    "colored headlights, tinted headlights, yellow headlights, painted wheels, body-colored wheels, "
    "color bleed onto glass, color bleed onto tires, color cast on chrome, recolored license plate, "
    "unchanged body color, original paint, identical to input, no color change, "
    "blurry, distorted, different car, different camera angle",
)

assert IMAGE_GUIDANCE >= 1.8, IMAGE_GUIDANCE
assert RESOLUTION == 512, RESOLUTION
print(f"Using image_guidance={IMAGE_GUIDANCE}  guidance={GUIDANCE}  steps={STEPS}  res={RESOLUTION}")

def enforce_square(img, size=RESOLUTION):
    img = img.convert("RGB")
    w, h = img.size
    side = min(w, h)
    left, top = (w - side) // 2, (h - side) // 2
    img = img.crop((left, top, left + side, top + side))
    return img.resize((size, size), Image.Resampling.LANCZOS)

t0 = time.perf_counter()
pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, safety_checker=None, requires_safety_checker=False,
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()
print(f"Loaded in {time.perf_counter()-t0:.1f}s")

In [ ]:
import shutil
from datetime import datetime, timezone
import numpy as np

OUT = Path("synthetic-pairs")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()
(OUT / "befores").mkdir()
(OUT / "afters").mkdir()

jobs = json.loads(jobs_path.read_text())
jobs = [j for j in jobs if j.get("modification_category") != "wheel_style"]
assert len(jobs) == 20, f"Expected 20 paint/wrap jobs, got {len(jobs)}"
print("Jobs:", len(jobs), "| categories:", {j["modification_category"] for j in jobs})

metadata, timings = [], []
SEED = 42

for i, job in enumerate(jobs):
    # Canonical IDs match local pipeline: 0001..0020 (NOT 0000..0019)
    pair_id = f"{i+1:04d}"
    before_raw = Image.open(PACK_DIR / job["before_path"]).convert("RGB")
    before = enforce_square(before_raw, RESOLUTION)

    print(f"[{i+1}/{len(jobs)}] #{pair_id} — {job['modification_category']}: {job['modification_change']}")
    t0 = time.perf_counter()
    gen = torch.Generator(device="cuda").manual_seed(SEED + i)
    call = dict(
        prompt=job["edit_prompt"],
        negative_prompt=NEGATIVE,
        image=before,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        image_guidance_scale=IMAGE_GUIDANCE,
        generator=gen,
    )
    try:
        out = pipe(height=RESOLUTION, width=RESOLUTION, **call)
    except TypeError:
        out = pipe(**call)
    after = out.images[0].convert("RGB")
    if after.size != (RESOLUTION, RESOLUTION):
        print(f"  WARN: model returned {after.size}; resizing to {RESOLUTION}x{RESOLUTION}")
        after = after.resize((RESOLUTION, RESOLUTION), Image.Resampling.LANCZOS)
    elapsed = time.perf_counter() - t0
    timings.append(elapsed)

    assert before.size == after.size == (RESOLUTION, RESOLUTION), (before.size, after.size)
    delta = float(np.abs(np.asarray(before, dtype=np.float32) - np.asarray(after, dtype=np.float32)).mean())
    flag = "NEAR_ZERO" if delta < 8 else ("STRONG" if delta > 35 else "ok")
    print(f"  {elapsed:.1f}s | size={after.size} | mean_abs_delta={delta:.1f} {flag}")

    bn, an = f"{pair_id}_before.jpg", f"{pair_id}_after.jpg"
    before.save(OUT / bn, quality=92)
    after.save(OUT / an, quality=92)
    before.save(OUT / "befores" / bn, quality=92)
    after.save(OUT / "afters" / an, quality=92)
    assert (OUT / bn).exists() and (OUT / an).exists(), (bn, an)

    # Schema required by pipeline/preview_synthetic.py
    metadata.append({
        "pair_id": pair_id,
        "id": pair_id,
        "title": f"Synthetic {job['modification_category']}: {job['modification_change']}",
        "source_type": "synthetic_compcars",
        "source_image": job.get("source_rel_path"),
        "source_viewpoint": job.get("source_viewpoint"),
        "source_bbox": job.get("source_bbox"),
        "modification_category": job["modification_category"],
        "modification_change": job["modification_change"],
        "edit_prompt": job["edit_prompt"],
        "instruction": job["edit_prompt"],
        "edit_model": MODEL_ID,
        "edit_backend": "instruct_pix2pix",
        "edit_steps": STEPS,
        "edit_resolution": RESOLUTION,
        "image_guidance_scale": IMAGE_GUIDANCE,
        "guidance_scale": GUIDANCE,
        "needs_localized_edit": False,
        "mean_abs_pixel_delta": round(delta, 2),
        "generation_seconds": round(elapsed, 2),
        "filenames": {"before": bn, "after": an},
        "dimensions": {"before": list(before.size), "after": list(after.size)},
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "review_status": "pending",
        "batch_tag": settings.get("batch_tag", "paint_wrap_retune_v2"),
    })

(OUT / "metadata.json").write_text(json.dumps(metadata, indent=2))
with open(OUT / "metadata.jsonl", "w") as f:
    for row in metadata:
        f.write(json.dumps(row) + "\n")

# Hard verify on-disk layout before export
flat_befores = sorted(OUT.glob("*_before.jpg"))
flat_afters = sorted(OUT.glob("*_after.jpg"))
assert len(flat_befores) == 20, f"Expected 20 flat befores, got {len(flat_befores)}"
assert len(flat_afters) == 20, f"Expected 20 flat afters, got {len(flat_afters)}"
assert (OUT / "metadata.json").exists()
for row in metadata:
    assert "pair_id" in row and "filenames" in row
    assert (OUT / row["filenames"]["before"]).exists()
    assert (OUT / row["filenames"]["after"]).exists()

avg = sum(timings) / len(timings)
size_ok = all(m["dimensions"]["before"] == m["dimensions"]["after"] == [512, 512] for m in metadata)
near_zero = sum(1 for m in metadata if m["mean_abs_pixel_delta"] < 8)
print(f"\nDone: {len(metadata)} pairs | avg {avg:.1f}s | all 512x512={size_ok} | near-zero flags={near_zero}")
print(f"On disk: {len(flat_befores)} befores + {len(flat_afters)} afters + metadata.json")
assert size_ok, "SIZE BUG: not every pair is 512x512"

In [ ]:
from IPython.display import display, HTML

html = ["<h3>Spot-check (first 4 pairs)</h3><div style='display:grid;gap:16px'>"]
for row in metadata[:4]:
    html.append(
        (
            "<div><b>#%s</b> %s — %s (delta=%.1f)</div>"
            "<div style='display:flex;gap:8px'>"
            "<img src='synthetic-pairs/%s' width='256'>"
            "<img src='synthetic-pairs/%s' width='256'>"
            "</div>"
        ) % (
            row["pair_id"], row["modification_category"], row["modification_change"],
            row["mean_abs_pixel_delta"],
            row["filenames"]["before"], row["filenames"]["after"],
        )
    )
html.append("</div>")
display(HTML("".join(html)))

In [ ]:
from google.colab import files

# Explicit zip — do NOT zip only afters/. Flat layout at zip root for preview_synthetic.py.
ZIP_PATH = Path("synthetic-pairs-v2.zip")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

required = ["metadata.json", "metadata.jsonl"]
for i in range(1, 21):
    pid = f"{i:04d}"
    required += [f"{pid}_before.jpg", f"{pid}_after.jpg"]

missing = [name for name in required if not (OUT / name).exists()]
assert not missing, f"Refusing to export — missing files: {missing[:10]}{'...' if len(missing)>10 else ''}"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in required:
        zf.write(OUT / name, arcname=name)
    # Optional copies (harmless if present)
    for sub in ("befores", "afters"):
        d = OUT / sub
        if d.exists():
            for f in sorted(d.glob("*.jpg")):
                zf.write(f, arcname=f"{sub}/{f.name}")

names = zipfile.ZipFile(ZIP_PATH).namelist()
n_before = sum(1 for n in names if n.endswith("_before.jpg") and "/" not in n)
n_after = sum(1 for n in names if n.endswith("_after.jpg") and "/" not in n)
has_meta = "metadata.json" in names
print(f"Zip: {ZIP_PATH.resolve()} ({ZIP_PATH.stat().st_size/1e6:.2f} MB)")
print(f"Flat befores={n_before} afters={n_after} metadata.json={has_meta}")
print("Sample entries:", names[:8], "...")
assert n_before == 20 and n_after == 20 and has_meta, (
    f"EXPORT BUG: zip incomplete (befores={n_before}, afters={n_after}, meta={has_meta})"
)

print("Downloading synthetic-pairs-v2.zip — unzip into EMPTY dataset/synthetic-pairs/ on your Mac")
files.download(str(ZIP_PATH))